# ML Model Analysis

- **XGBoost**: feature importance, precision/recall across TP-probability thresholds
- **HMM**: state transition matrix, mean returns per state, regime duration

In [ ]:
import sys
import numpy as np
import pandas as pd
sys.path.insert(0, '../src')

from data.fetcher import fetch_data
from squeeze.indicators import compute_indicators
from squeeze.signals import generate_base_signal
from ml.xgboost_scorer import label_signals, train_xgb, FEATURE_COLS
from ml.hmm_regime import fit_hmm, hmm_bull_states

SYMBOL = 'PSO'
df_raw = fetch_data(SYMBOL)
df     = compute_indicators(df_raw)
df     = generate_base_signal(df)

In [ ]:
# XGBoost feature importance
labeled      = label_signals(df, df['base_signal'])
model, scaler = train_xgb(labeled)

importances = pd.Series(model.feature_importances_, index=FEATURE_COLS)
importances.sort_values(ascending=False).plot(kind='bar', title='XGBoost Feature Importance')

In [ ]:
# HMM state analysis
log_r = df['log_ret'].values
hmm   = fit_hmm(log_r)
bulls = hmm_bull_states(hmm, log_r)

print('HMM means (log-return per state):', hmm.means_.flatten())
print('Transition matrix:\n', hmm.transmat_)
print(f'Bull state fraction: {bulls.mean():.2%}')